<a href="https://colab.research.google.com/github/ericb42/electric-vehicles/blob/main/GB885_Assignment_6_Brauer_E.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Electric Vehicle Analysis for the State of Washington

##Step 1. Understand the Problem

Objective:

> Analyze the relationship between average county income and the rate of electric vehicle (EV) registration in the state of Washington.
>
> A secondary objective is to investigate whether average county income is associated with the distribution of EV makes and models.
>
> Identifying these relationships could help EV manufacturers prioritize marketing efforts and identify counties that may represent strong opportunities for dealership expansion, service center placement, and future charging infrastructure investments.

In [ ]:
# Import Python Libraries
import pandas as pd
import numpy as np
import io              # for loading files

##Step 2. Data Acquisition and Consolidation

Data will be combined from three data sets.
 - [Washington EV registration data](https://data.wa.gov/Transportation/Electric-Vehicle-Population-Data/f6w7-q2d2/about_data)
 - [US Census SAIPE State & County Estimates for 2024](https://www.census.gov/data/datasets/2024/demo/saipe/2024-state-and-county.html) (provides income)
 - [Washington Population Change by County, 2020-2026](https://data.wa.gov/demographics/WAOFM-April-1-Population-Change-and-Rank-by-County/nde6-xvwf/about_data)

In [ ]:
# Upload data
from google.colab import files

# Washington EV registration data
ev_data_uploaded = files.upload()

# US Census income data
income_data_uploaded = files.upload()

# Washington population data
population_data_uploaded = files.upload()

In [ ]:
# Import data

# Washington EV registration data
ev_df = pd.read_csv(io.BytesIO(ev_data_uploaded['ev_vehicle_population_data.csv']))

# US Census income data
income_df = pd.read_csv(io.BytesIO(income_data_uploaded['us_all_states_and_counties_income.csv']), skiprows = [0,1], header = 1)

# Washington poplation data
population_df = pd.read_csv(io.BytesIO(population_data_uploaded['population_data.csv']))

### Set Pandas options and preview the data

In [ ]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [ ]:
# preview EV data
ev_df.head()

In [ ]:
# preview income data
income_df.head()

In [ ]:
# preview population data
population_df.head()

### Unwanted Observations
We are only looking at the state of Washington.

Our population_df and ev_df are already structured to only contain Washington data, but the income_df contains information from all US states.

Because we will be combining the data sets based on county name, it will be simpler to eliminate all non-Washington data from the income_df before combining.

In [ ]:
# filter income_df data to only include observations from Washington (Postal Code = WA)

# create the filter
state_filter = income_df['Postal Code'] == 'WA'
# apply the filter
wa_income_df = income_df[state_filter]

We will use the county names to combine the data, so we will also need to standardize this information because wa_income_df contains the word "County" at the end of each county name.

Note: This will leave Washington in the Name column, which refers to the entire state, not a county, but this will not be an issue since there is not Washington County, so the state level data will not be included in our merge.

In [ ]:
# standardize merge column (the county name) in wa_income_df
wa_income_df.loc[:, 'Name'] = wa_income_df['Name'].str.replace(' County', '').str.strip()

In [ ]:
wa_income_df.head()

### Consolidate three dataframes

In [ ]:
# merge ev_df with wa_income_df

ev_and_income_df = pd.merge(ev_df, wa_income_df, left_on = 'County', right_on = 'Name', how = 'left')

# check our work
ev_and_income_df.head()

In [ ]:
# merge new ev_and_income_df with population_df
full_df = pd.merge(ev_and_income_df, population_df, left_on = 'County', right_on = 'County', how = 'left')

# check our work
full_df.head()

##Step 3. Inspect the Data
Look For:

- Unwanted Observations
- Unwanted Features
- Incorrect Data Formats or DataTypes
- Duplicate Values
- Missing Values
- Erroneous Values
- Outliers

### Inspect Data Characteristics

In [ ]:
full_df.head()

### Unwanted observations
Washington has records for vehicles that are registered/licensed in other states, but we're only intereseted in vehicles registered in Washington.

In [ ]:
# filter income_df data to only include observations from Washington (Postal Code = WA)

# create the filter
state_filter = full_df['State_x'] == 'WA'
# apply the filter
full_df = full_df[state_filter]

In [ ]:
full_df.info()

List of features to remove:
 - None of the 'Rank' features are needed

   - RANK_2020
   - RANK_2021
   - RANK_2022
   - RANK_2023
   - RANK_2024
   - RANK_2025
   - RANK_2026

 - None of the 'PC' features are needed

   - PC_20-21
   - PC_21-22
   - PC_22-23
   - PC_23-24
   - PC_24-25
   - PC_25-26

 - None of the 'NC' features are needed
   - NC_20-21
   - NC_21-22
   - NC_22-23
   - NC_23-24
   - NC_24-25
   - NC_25-26

 - None of the '90%' features are needed

   - 90% CI Lower Bound
   - 90% CI Upper Bound
   - 90% CI Lower Bound.1
   - 90% CI Upper Bound.1
   - 90% CI Lower Bound.2
   - 90% CI Upper Bound.2
   - 90% CI Lower Bound.3
   - 90% CI Upper Bound.3
   - 90% CI Lower Bound.4
   - 90% CI Upper Bound.4
   - 90% CI Lower Bound.5
   - 90% CI Upper Bound.5
   - 90% CI Lower Bound.6
   - 90% CI Upper Bound.6
   - 90% CI Lower Bound.7
   - 90% CI Upper Bound.7
   - 90% CI Lower Bound.8
   - 90% CI Upper Bound.8

 - None of the 'Poverty' features are needed

   - Poverty Estimate, All Ages
   - Poverty Percent, All Ages
   - Poverty Estimate, Age 0-17
   - Poverty Percent, Age 0-17
   - Poverty Estimate, Age 5-17
   - Poverty Percent, Age 5-17
   - Poverty Estimate, Age 0-4
   - Poverty Percent, Age 0-4

 - Longitude
 - Latitude
 - Legislative District
 - Vehicle Location
 - Electric Utility
 - 2020 Census Tract
 - State FIPS Code
 - County FIPS Code
 - Electric Range

 - These are duplicated features resulting from the merge
   - Postal Code_y
   - Name
   - COUNTY_NAME
   - State_y

 - We can remove 'VIN (1-10)' because we have 'DOL Vehicle ID'
 - We only need one year of population data, so we'll select 2024 since that's the year of the income data. We can drop the population data for the other years.
   - POP_2020
   - POP_2021
   - POP_2022
   - POP_2023
   - POP_2025
   - POP_2026

In [ ]:
# remove the features listed above
remove_features = ['RANK_2020',
                   'RANK_2021',
                   'RANK_2022',
                   'RANK_2023',
                   'RANK_2024',
                   'RANK_2025',
                   'RANK_2026',
                   'PC_20-21',
                   'PC_21-22',
                   'PC_22-23',
                   'PC_23-24',
                   'PC_24-25',
                   'PC_25-26',
                   'NC_20-21',
                   'NC_21-22',
                   'NC_22-23',
                   'NC_23-24',
                   'NC_24-25',
                   'NC_25-26',
                   '90% CI Lower Bound',
                   '90% CI Upper Bound',
                   '90% CI Lower Bound.1',
                   '90% CI Upper Bound.1',
                   '90% CI Lower Bound.2',
                   '90% CI Upper Bound.2',
                   '90% CI Lower Bound.3',
                   '90% CI Upper Bound.3',
                   '90% CI Lower Bound.4',
                   '90% CI Upper Bound.4',
                   '90% CI Lower Bound.5',
                   '90% CI Upper Bound.5',
                   '90% CI Lower Bound.6',
                   '90% CI Upper Bound.6',
                   '90% CI Lower Bound.7',
                   '90% CI Upper Bound.7',
                   '90% CI Lower Bound.8',
                   '90% CI Upper Bound.8',
                   'Poverty Estimate, All Ages',
                   'Poverty Percent, All Ages',
                   'Poverty Estimate, Age 0-17',
                   'Poverty Percent, Age 0-17',
                   'Poverty Estimate, Age 5-17 in Families',
                   'Poverty Percent, Age 5-17 in Families',
                   'Poverty Estimate, Age 0-4',
                   'Poverty Percent, Age 0-4',
                   'Longitude',
                   'Latitude',
                   'Legislative District',
                   'Vehicle Location',
                   'Electric Utility',
                   '2020 Census Tract',
                   'State FIPS Code',
                   'County FIPS Code',
                   'Electric Range',
                   'Postal Code_y',
                   'Name',
                   'COUNTY_NAME',
                   'State_y',
                   'VIN (1-10)',
                   'POP_2020',
                   'POP_2021',
                   'POP_2022',
                   'POP_2023',
                   'POP_2025',
                   'POP_2026']

# drop the features
clean_data_df = full_df.drop(remove_features, axis = 1)

In [ ]:
clean_data_df.head()

In [ ]:
clean_data_df.info()

List of data type/format corrections:
 - Postal Code_x should be int64
 - Electric Range should be int64
 - Median Household Income should be int64
 - POP_2024 should be int64


In [ ]:
# List of features to convert to numeric
to_numeric = ['Postal Code_x', 'Median Household Income', 'POP_2024']

# unique values in features we expect to be numeric
for column in to_numeric:
  print(column)
  print(clean_data_df[column].unique())

In [ ]:
# remove commas from Median Household Income and POP_2024 so that it can be converted
clean_data_df['Median Household Income'] = clean_data_df['Median Household Income'].str.replace(',', '')
clean_data_df['POP_2024'] = clean_data_df['POP_2024'].str.replace(',', '')

clean_data_df.head()

In [ ]:
# convert features to numeric
for column in to_numeric:
  clean_data_df[column] = pd.to_numeric(clean_data_df[column], errors = 'coerce')

In [ ]:
clean_data_df.head()

In [ ]:
clean_data_df.info()

In [ ]:
# convert float to int for Postal Code_x
clean_data_df['Postal Code_x'] = clean_data_df['Postal Code_x'].astype('int')

In [ ]:
clean_data_df.info()

### Inspect for null values

In [ ]:
clean_data_df.shape

In [ ]:
# Traditional null values
clean_data_df.isnull().sum()

In [ ]:
# Non-traditional categorical data

# list of cateogrical variables in dataframe
cat_var = list(clean_data_df.select_dtypes(include = ['object']).columns)

# view unique values for each categorical variable
for column in cat_var:
  print(column)
  print(clean_data_df[column].unique())

In [ ]:
# non-traditional null values: numerical data
clean_data_df.describe()

### Inspect for duplicate values

In [ ]:
clean_data_df.duplicated().sum()

### Check for Erroneous Values
Based on previous numerical and categorical checks, I do not see any values that appear to be out of order.

## Step 4: Data Cleaning and Preparation

Data type/formatting, unwanted observations, and unwanted features were addressed in Step 3.

Since no missing values, erroneous values, outliers, or duplicates were identified during inspection, no additional cleaning is needed.